# 03 — Seasonal Analysis
Seasonal patterns in EU gas storage injection and withdrawal.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f"Project root: {_c}")
        break

DATA = Path("data/processed")
df = pd.read_parquet(DATA / "eu_aggregate_full.parquet")
df.index = pd.to_datetime(df.index).tz_localize(None)
df = df.sort_index()
df["year"] = df.index.year
df["month"] = df.index.month
df["doy"] = df.index.day_of_year
print(f"eu_aggregate_full.parquet: {df.shape}")
print(f"  {df.index[0].date()} -> {df.index[-1].date()}")


## 1. Average Injection Rate by Month
Active injection days only (injection > 0), April–October.

In [ ]:
# Active injection days: injection > 0, Apr-Oct
inj = df[(df["injection"] > 0) & (df["month"].isin(range(4, 11)))].copy()
monthly_inj = inj.groupby("month")["injection"].agg(["mean","std","count"])
monthly_inj.index = [pd.Timestamp(2020, m, 1).strftime("%b") for m in monthly_inj.index]
monthly_inj.columns = ["Mean (GWh/day)", "Std (GWh/day)", "Active Days"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly_inj.index,
    y=monthly_inj["Mean (GWh/day)"],
    error_y=dict(type="data", array=monthly_inj["Std (GWh/day)"], visible=True),
    marker_color="#27AE60", name="Mean injection"
))
fig.update_layout(
    title="Average Daily Injection Rate by Month (active injection days, Apr-Oct)",
    xaxis_title="Month", yaxis_title="GWh/day",
    height=420, template="plotly_white"
)
fig.show()
print(monthly_inj.round(0).to_string())


## 2. Average Withdrawal Rate by Month
Active withdrawal days only (withdrawal > 0), October–March.

In [ ]:
# Active withdrawal days: withdrawal > 0, Oct-Mar
wd = df[(df["withdrawal"] > 0) & (df["month"].isin([10,11,12,1,2,3]))].copy()
month_order = [10, 11, 12, 1, 2, 3]
monthly_wd = (
    wd.groupby("month")["withdrawal"]
    .agg(["mean","std","count"])
    .reindex(month_order)
)
monthly_wd.index = [pd.Timestamp(2020, m, 1).strftime("%b") for m in month_order]
monthly_wd.columns = ["Mean (GWh/day)", "Std (GWh/day)", "Active Days"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly_wd.index,
    y=monthly_wd["Mean (GWh/day)"],
    error_y=dict(type="data", array=monthly_wd["Std (GWh/day)"], visible=True),
    marker_color="#E74C3C", name="Mean withdrawal"
))
fig.update_layout(
    title="Average Daily Withdrawal Rate by Month (active withdrawal days, Oct-Mar)",
    xaxis_title="Month", yaxis_title="GWh/day",
    height=420, template="plotly_white"
)
fig.show()
print(monthly_wd.round(0).to_string())


## 3. Seasonal Decomposition
Additive decomposition of daily fill % into trend, seasonal, and residual components.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from plotly.subplots import make_subplots

fill_daily = df["full"].resample("D").interpolate("linear").dropna()
decomp = seasonal_decompose(fill_daily, model="additive", period=365, extrapolate_trend="freq")

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=["Observed", "Trend", "Seasonal", "Residual"],
    shared_xaxes=True, vertical_spacing=0.06
)
pairs = [
    (decomp.observed, "#2E75B6"),
    (decomp.trend,    "#E67E22"),
    (decomp.seasonal, "#27AE60"),
    (decomp.resid,    "#95A5A6"),
]
for row, (series, color) in enumerate(pairs, start=1):
    fig.add_trace(
        go.Scatter(x=series.index, y=series.values,
                   line=dict(color=color, width=1), showlegend=False),
        row=row, col=1
    )
fig.update_layout(
    title="Seasonal Decomposition of EU Storage Fill % (additive, period = 365 days)",
    height=780, template="plotly_white"
)
fig.show()
print(f"Seasonal amplitude: {decomp.seasonal.max() - decomp.seasonal.min():.1f} pp")
print(f"Trend range:        {decomp.trend.min():.1f}% -> {decomp.trend.max():.1f}%")


## 4. Day-of-Year Percentile Bands
P10 / P50 / P90 fill % across all historical years, with current year overlay.

In [ ]:
bands = df.groupby("doy")["full"].quantile([0.10, 0.50, 0.90]).unstack(level=1)
bands.columns = ["P10", "P50", "P90"]

current_year = df["year"].max()
this_year = df[df["year"] == current_year].set_index("doy")["full"]

fig = go.Figure()
# P10-P90 shaded band
fig.add_trace(go.Scatter(
    x=list(bands.index) + list(bands.index[::-1]),
    y=list(bands["P90"]) + list(bands["P10"][::-1]),
    fill="toself", fillcolor="rgba(46,117,182,0.15)",
    line=dict(color="rgba(0,0,0,0)"), name="P10-P90 band"
))
fig.add_trace(go.Scatter(
    x=bands.index, y=bands["P50"],
    name="Historical Median (P50)",
    line=dict(color="#2E75B6", width=2, dash="dash")
))
fig.add_trace(go.Scatter(
    x=this_year.index, y=this_year.values,
    name=str(current_year),
    line=dict(color="#E74C3C", width=2.5)
))
fig.add_hline(y=90, line_dash="dot", line_color="black", line_width=1,
              annotation_text="90% Target")
tick_vals = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
tick_text  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig.update_layout(
    title="EU Storage Fill % — Day-of-Year Percentile Bands (P10 / P50 / P90, all years)",
    xaxis=dict(title="Day of Year", tickvals=tick_vals, ticktext=tick_text),
    yaxis=dict(title="Fill (%)", range=[0, 110]),
    height=460, template="plotly_white"
)
fig.show()
